# Recomendador Colaborativo Item-Item

Genera recomendaciones de productos usando filtrado colaborativo basado en similitud coseno item-item.

Replica `src/train_recommender.py` (top 10% global) y `src/train_recommender_by_client.py` (top 5 por cliente).

**Proceso:**
1. Cargar transacciones preprocesadas (`filtered_agg_sales_for_rec.parquet`).
2. Construir matriz dispersa usuario-item.
3. Calcular similitud coseno entre productos.
4. Generar recomendaciones por cliente.

---

## 1. Setup e Importaciones

In [1]:
import os
import sys
from pathlib import Path

# Fijar directorio de trabajo en la raíz del proyecto
# (las rutas de params.yaml son relativas a la raíz)
os.chdir(Path("..").resolve())
sys.path.insert(0, "src")

import logging
import pandas as pd
import numpy as np

from config import load_config, processed_path
from data_io import load_parquet, load_product_catalog
from collaborative import create_sparse_matrix, compute_item_similarity, recommend_for_client

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")

cfg = load_config("params.yaml")
rec_cfg = cfg["recommendations_item_item"]
print(f"Directorio de trabajo: {os.getcwd()}")
print("Config cargada")

Directorio de trabajo: /Users/juandavidrincon/Documents/euro_nuevo_nov/euro
Config cargada


## 2. Carga de Datos

`filtered_agg_sales_for_rec.parquet` es generado por `src/preprocess.py`:  
transacciones agregadas (cliente, producto, día) de clientes con patrones regulares.

In [2]:
proc = processed_path(cfg)
transactions = load_parquet(
    proc / rec_cfg["transaction_data_processed_file"],
    "Transacciones",
)
productos = load_product_catalog(proc / cfg["data"]["products_file"])

# Renombrar id_client → client (convención downstream usada por collaborative.py)
if "id_client" in transactions.columns and "client" not in transactions.columns:
    transactions = transactions.rename(columns={"id_client": "client"})

print(f"Transacciones: {len(transactions):,} filas")
print(f"Clientes: {transactions['client'].nunique():,}")
print(f"Productos: {transactions['product'].nunique():,}")
transactions.head()

INFO - Transacciones cargado: 2134222 filas desde data/processed/filtered_agg_sales_for_rec.parquet
INFO - Catálogo cargado: 26534 productos únicos


Transacciones: 2,134,222 filas
Clientes: 6,426
Productos: 15,387


,date_sale,client,product,quantity,amount_paid
0,2025-07-01,1000088250,328582,1.00,1022.0
1,2025-07-01,1000088250,81289,0.16,562.0
2,2025-07-02,1000088250,333211,1.00,14306.0
3,2025-07-03,1000088250,250234,1.00,7577.0
4,2025-07-03,1000088250,329686,1.00,2307.0


## 3. Filtrado de Productos Top 10%

Reduce el espacio de productos a los más frecuentes globalmente para mejorar la calidad de similitud.  
Replica la lógica de `src/train_recommender.py`.

In [3]:
# Frecuencia de cada producto
product_freq = transactions.groupby("product").size().reset_index(name="freq")
threshold = product_freq["freq"].quantile(0.90)

top_products = product_freq[product_freq["freq"] >= threshold]["product"]
transactions_top = transactions[transactions["product"].isin(top_products)].copy()

print(f"Umbral de frecuencia (p90): {threshold:.0f}")
print(f"Productos top 10%: {len(top_products):,} de {len(product_freq):,}")
print(f"Transacciones tras filtro: {len(transactions_top):,}")

Umbral de frecuencia (p90): 224
Productos top 10%: 1,546 de 15,387
Transacciones tras filtro: 1,584,567


## 4. Matriz Dispersa Usuario-Item

Construye la matriz donde cada celda (i, j) cuenta cuántas veces el cliente i compró el producto j.  
Usa `src/collaborative.create_sparse_matrix()`.

In [4]:
sparse_matrix, mappings = create_sparse_matrix(transactions_top)

if sparse_matrix is not None:
    print(f"Matriz: {sparse_matrix.shape[0]:,} usuarios x {sparse_matrix.shape[1]:,} productos")
    print(f"Interacciones no-cero: {sparse_matrix.nnz:,}")
    density = sparse_matrix.nnz / (sparse_matrix.shape[0] * sparse_matrix.shape[1])
    print(f"Densidad: {density:.4f}")
else:
    print("No hay datos suficientes para construir la matriz.")

INFO - Matriz 6425 usuarios x 1546 productos | 566296 interacciones | densidad: 0.0570


Matriz: 6,425 usuarios x 1,546 productos
Interacciones no-cero: 566,296
Densidad: 0.0570


## 5. Similitud Item-Item (Coseno)

Calcula la similitud coseno entre todos los pares de productos.  
Usa `src/collaborative.compute_item_similarity()`.

In [5]:
similarity = compute_item_similarity(sparse_matrix)

if similarity is not None:
    print(f"Matriz de similitud: {similarity.shape}")
    print(f"Pares con similitud > 0: {similarity.nnz:,}")
else:
    print("No se pudo calcular similitud.")

INFO - Calculando similitud Item-Item (coseno)...
INFO - Similitud calculada: (1546, 1546)


Matriz de similitud: (1546, 1546)
Pares con similitud > 0: 2,342,492


## 6. Recomendaciones por Cliente

Para cada cliente, pondera la similitud de productos no comprados con los comprados,  
usando la frecuencia de compra como peso. Usa `src/collaborative.recommend_for_client()`.

In [6]:
n_recs = rec_cfg["num_recommendations"]
product_map = mappings["product_map"]
user_map = mappings["user_map"]

# Recomendar para los primeros 5 clientes como ejemplo
sample_clients = list(user_map.items())[:5]

for idx, client_id in sample_clients:
    recs = recommend_for_client(
        client_idx=idx,
        sparse_matrix=sparse_matrix,
        similarity_matrix=similarity,
        product_map=product_map,
        n_recs=n_recs,
        return_scores=True,
    )
    # Enriquecer con descripciones
    recs_df = pd.DataFrame(recs)
    if not recs_df.empty:
        recs_df = recs_df.merge(
            productos, left_on="recommended_product", right_on="product", how="left"
        )
        print(f"\nCliente {client_id} — Top {n_recs} recomendaciones:")
        for _, row in recs_df.iterrows():
            desc = row.get("description", row["recommended_product"])
            print(f"  {desc} (score: {row['recommendation_score']:.3f})")
    else:
        print(f"\nCliente {client_id} — Sin recomendaciones (historial limitado)")


Cliente 1000088250 — Top 5 recomendaciones:
  SOPA DEL DIA X 300GR (score: 9.582)
  JUGO NATURAL X 12 OZ MENU (score: 9.494)
  EMPANADA DE CARNE  x 100GR (score: 8.904)
  PASTEL DE AREQUIPE MULTIPAN *85GR (score: 8.505)
  MENU EMPLEADOS EURO (score: 8.400)

Cliente 1000088412 — Top 5 recomendaciones:
  GALLETAS FESTIVAL LIMON *50GR (score: 7.517)
  GALLETA DUX SALADITA BOLS *52.8GR (score: 5.864)
  CHOCOLATINA GOL X 28 GR (score: 5.814)
  SALCHIPAPAS  x 250GR EURO (score: 5.699)
  nan (score: 5.134)

Cliente 1000088935 — Top 5 recomendaciones:
  TOMATE CHONTO (score: 15.497)
  PLATANO (score: 14.230)
  JUGO NATURAL X 12 OZ MENU (score: 13.992)
  ZANAHORIA (score: 13.879)
  SOPA DEL DIA X 300GR (score: 13.766)

Cliente 1000192661 — Top 5 recomendaciones:
  PASTEL DE POLLO  x 150GR (score: 35.164)
  COMBO HAMBURGUESA MIXTA PANTALLAS (score: 33.621)
  AGUA POTABLE TRATADA EUROMAX *1000ML (score: 30.695)
  PALITO DE QUESO MULTIPAN *100GR (score: 29.501)
  nan (score: 27.625)

Cliente 1000

## 7. Generación Masiva (Pre-cómputo)

En producción se pre-calculan las recomendaciones para **todos** los clientes  
y se guardan en `precomputed_item_item_recs.parquet`.

In [7]:
# Pre-calcular para todos los clientes
all_recs = []
for idx, client_id in user_map.items():
    recs = recommend_for_client(
        client_idx=idx,
        sparse_matrix=sparse_matrix,
        similarity_matrix=similarity,
        product_map=product_map,
        n_recs=n_recs,
        return_scores=True,
    )
    for rank, rec in enumerate(recs, 1):
        all_recs.append({
            "client": client_id,
            "recommended_product": rec["recommended_product"],
            "recommendation_score": rec["recommendation_score"],
            "recommendation_rank": rank,
        })

df_all_recs = pd.DataFrame(all_recs)
print(f"Total recomendaciones generadas: {len(df_all_recs):,}")
print(f"Clientes con recomendaciones: {df_all_recs['client'].nunique():,}")
df_all_recs.head(10)

Total recomendaciones generadas: 32,125
Clientes con recomendaciones: 6,425


,client,recommended_product,recommendation_score,recommendation_rank
0,1000088250,82789,9.581769,1
1,1000088250,88042,9.494219,2
2,1000088250,110037,8.903730,3
3,1000088250,115569,8.505230,4
4,1000088250,102491,8.400460,5
5,1000088412,89287,7.516520,1
6,1000088412,332271,5.863796,2
7,1000088412,326918,5.813707,3
8,1000088412,75076,5.699131,4
9,1000088412,333211,5.133769,5


---

**Para ejecutar en producción:**

```bash
# Top 10% global
uv run python src/train_recommender.py --config params.yaml

# Top 5 por cliente
uv run python src/train_recommender_by_client.py --config params.yaml
```

**Para unir recomendaciones con predicciones:**

```bash
uv run python src/get_recommendations.py \
    --input_file predictions/predictions_client.csv \
    --output_file predictions/preds_with_recs.csv \
    --include_clustering
```